In [ ]:
import os
import librosa
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

DATASET_PATH = r"music_dataset\Data\genres_original"
OUTPUT_CSV = "music_5sec_features.csv"
SLICE_DURATION = 5  
SAMPLE_RATE = 22050 

def extract_features_from_slice(y_slice, sr, label):
    features = {}
    # MFCCs (13 Mean, 13 Variance)
    mfccs = librosa.feature.mfcc(y=y_slice, sr=sr, n_mfcc=13)
    for i in range(13):
        features[f'mfcc_{i+1}_mean'] = np.mean(mfccs[i])
        features[f'mfcc_{i+1}_var'] = np.var(mfccs[i])
    # Chroma (Harmony) (12 Mean, 12 Variance)
    chroma = librosa.feature.chroma_stft(y=y_slice, sr=sr)
    for i in range(12):
        features[f'chroma_{i+1}_mean'] = np.mean(chroma[i])
        features[f'chroma_{i+1}_var'] = np.var(chroma[i])
        
    # Mean and Variance
    cent = librosa.feature.spectral_centroid(y=y_slice, sr=sr)
    features['centroid_mean'] = np.mean(cent)
    features['centroid_var'] = np.var(cent)
    
    band = librosa.feature.spectral_bandwidth(y=y_slice, sr=sr)
    features['bandwidth_mean'] = np.mean(band)
    features['bandwidth_var'] = np.var(band)
    
    rolloff = librosa.feature.spectral_rolloff(y=y_slice, sr=sr)
    features['rolloff_mean'] = np.mean(rolloff)
    features['rolloff_var'] = np.var(rolloff)
    
    flatness = librosa.feature.spectral_flatness(y=y_slice)
    features['flatness_mean'] = np.mean(flatness)
    features['flatness_var'] = np.var(flatness)
    
    # Waveform Features
    zcr = librosa.feature.zero_crossing_rate(y_slice)
    features['zcr_mean'] = np.mean(zcr)
    features['zcr_var'] = np.var(zcr)
    
    rms = librosa.feature.rms(y=y_slice)
    features['rms_mean'] = np.mean(rms)
    features['rms_var'] = np.var(rms)
    
    # Rhythm
    tempo, _ = librosa.beat.beat_track(y=y_slice, sr=sr)
    features['tempo'] = float(tempo)
    
    features['label'] = label
    
    return features

def build_dataset():
    all_data = []
    
    samples_per_slice = SLICE_DURATION * SAMPLE_RATE
    
    for genre in os.listdir(DATASET_PATH):
        genre_folder = os.path.join(DATASET_PATH, genre)
        
        if not os.path.isdir(genre_folder):
            continue
            
        print(f"Processing genre: {genre}...")
            
        for filename in os.listdir(genre_folder):
            if filename.lower().endswith('.wav'):
                file_path = os.path.join(genre_folder, filename)        
                try:
                    y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
                    num_slices = len(y) // samples_per_slice
                    for i in range(num_slices):
                        start = i * samples_per_slice
                        end = start + samples_per_slice
                        y_slice = y[start:end]
                        slice_features = extract_features_from_slice(y_slice, sr, label=genre)
                        all_data.append(slice_features)
                        
                except Exception as e:
                    print(f"  -> Skipped {filename} due to error: {e}")
                    
    df = pd.DataFrame(all_data)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSUCCESS! Dataset saved as {OUTPUT_CSV}")
    print(f"Total Rows: {len(df)}")
    print(f"Total Columns: {len(df.columns)}")

if __name__ == "__main__":
    build_dataset()

Processing genre: blues...
Processing genre: classical...
Processing genre: country...
Processing genre: disco...
Processing genre: hiphop...
Processing genre: jazz...
  -> Skipped jazz.00054.wav due to error: 
Processing genre: metal...
Processing genre: pop...
Processing genre: reggae...
Processing genre: rock...

SUCCESS! Dataset saved as music_5sec_features.csv
Total Rows: 5985
Total Columns: 64


In [5]:
df = pd.read_csv('music_5sec_features.csv')

df.head()

,mfcc_1_mean,mfcc_1_var,mfcc_2_mean,mfcc_2_var,mfcc_3_mean,mfcc_3_var,mfcc_4_mean,mfcc_4_var,mfcc_5_mean,mfcc_5_var,...,rolloff_mean,rolloff_var,flatness_mean,flatness_var,zcr_mean,zcr_var,rms_mean,rms_var,tempo,label
0,-119.217600,2274.5337,124.338330,251.25555,-22.838790,328.88797,42.07474,158.041280,-7.011255,151.27107,...,3745.481364,9.667655e+05,0.004447,0.000036,0.083953,0.000825,0.124171,0.002805,123.046875,blues
1,-131.973660,3024.8772,116.727630,205.12883,-14.334999,158.68523,48.22651,200.175080,-4.449632,143.29967,...,3949.947103,6.724590e+05,0.003909,0.000014,0.075758,0.000580,0.124230,0.003497,123.046875,blues
2,-109.010670,2448.1730,134.205020,296.46460,-19.396326,156.30157,40.78598,118.761720,-13.618481,131.64221,...,3488.927205,7.501875e+05,0.003425,0.000010,0.069698,0.000297,0.141030,0.002071,123.046875,blues
3,-102.702530,2601.4660,113.554344,399.55685,-16.050467,189.84277,43.88332,146.786790,-10.231324,118.03165,...,4313.170369,1.146375e+06,0.007680,0.000082,0.092857,0.001245,0.128789,0.003560,129.199219,blues
4,-107.959145,2039.5167,125.162780,237.26869,-20.689390,275.86188,41.46085,114.088264,-11.114463,189.38951,...,3813.171387,8.246002e+05,0.004225,0.000030,0.073932,0.000296,0.140579,0.002559,123.046875,blues


In [6]:
df['label'].value_counts()

label
blues        600
pop          600
metal        600
reggae       600
rock         599
disco        599
classical    598
hiphop       598
country      597
jazz         594
Name: count, dtype: int64

In [8]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("Loading dataset...")
df = pd.read_csv('music_5sec_features.csv')

X = df.drop(columns=['label'])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Running K-Means Clustering...")
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)

df['Cluster'] = kmeans.fit_predict(X_scaled)

print("\n--- K-Means Results: True Labels vs. Machine Clusters ---")
comparison_table = pd.crosstab(
    df['label'], 
    df['Cluster'], 
    rownames=['True Folder Name'], 
    colnames=['K-Means Cluster']
)

print(comparison_table)

Loading dataset...
Running K-Means Clustering...

--- K-Means Results: True Labels vs. Machine Clusters ---
K-Means Cluster     0    1    2    3    4   5    6    7    8    9
True Folder Name                                                 
blues              52   38    8    0   33   2   25  232  210    0
classical           2  220   11    0    1  88  251   16    8    1
country             1   28   93    2   20   2   65  144  157   85
disco             121    0   73   11   55   0    4  139    5  191
hiphop             95    0   81  149  131   0    2   55    3   82
jazz                5  169   45    0    1  20  100  120   44   90
metal             509    0    3    0   23   0    1   53    0   11
pop                 1    7  102  226    0   3    4    0   14  243
reggae             11    4  162   85  233   2    8   31   37   27
rock               98    4   74    2   33   2   25  219   21  121


In [3]:
import os
import librosa
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

DATASET_PATH = r"music_dataset\Data\genres_original"
OUTPUT_CSV = "music_5sec_features(upd).csv"
SLICE_DURATION = 5  
SAMPLE_RATE = 22050 

def extract_features_from_slice(y_slice, sr, label, track_id):
    features = {}
    
    # Adding the integer ID as the first column
    features['track_id'] = track_id
    
    # MFCCs (13 Mean, 13 Variance)
    mfccs = librosa.feature.mfcc(y=y_slice, sr=sr, n_mfcc=13)
    for i in range(13):
        features[f'mfcc_{i+1}_mean'] = np.mean(mfccs[i])
        features[f'mfcc_{i+1}_var'] = np.var(mfccs[i])
        
    # Chroma (Harmony) (12 Mean, 12 Variance)
    chroma = librosa.feature.chroma_stft(y=y_slice, sr=sr)
    for i in range(12):
        features[f'chroma_{i+1}_mean'] = np.mean(chroma[i])
        features[f'chroma_{i+1}_var'] = np.var(chroma[i])
        
    # Spectral Features: Mean and Variance
    cent = librosa.feature.spectral_centroid(y=y_slice, sr=sr)
    features['centroid_mean'] = np.mean(cent)
    features['centroid_var'] = np.var(cent)
    
    band = librosa.feature.spectral_bandwidth(y=y_slice, sr=sr)
    features['bandwidth_mean'] = np.mean(band)
    features['bandwidth_var'] = np.var(band)
    
    rolloff = librosa.feature.spectral_rolloff(y=y_slice, sr=sr)
    features['rolloff_mean'] = np.mean(rolloff)
    features['rolloff_var'] = np.var(rolloff)
    
    flatness = librosa.feature.spectral_flatness(y=y_slice)
    features['flatness_mean'] = np.mean(flatness)
    features['flatness_var'] = np.var(flatness)
    
    # Waveform Features
    zcr = librosa.feature.zero_crossing_rate(y_slice)
    features['zcr_mean'] = np.mean(zcr)
    features['zcr_var'] = np.var(zcr)
    
    rms = librosa.feature.rms(y=y_slice)
    features['rms_mean'] = np.mean(rms)
    features['rms_var'] = np.var(rms)
    
    # Rhythm
    tempo, _ = librosa.beat.beat_track(y=y_slice, sr=sr)
    features['tempo'] = float(tempo[0])
    
    # The Answer Key
    features['label'] = label
    
    return features

def build_dataset():
    all_data = []
    
    samples_per_slice = SLICE_DURATION * SAMPLE_RATE
    
    # --- ADDED: This is your starting ID number ---
    current_song_id = 1 
    
    for genre in os.listdir(DATASET_PATH):
        genre_folder = os.path.join(DATASET_PATH, genre)
        
        if not os.path.isdir(genre_folder):
            continue
            
        print(f"Processing genre: {genre}...")
            
        for filename in os.listdir(genre_folder):
            if filename.lower().endswith('.wav'):
                file_path = os.path.join(genre_folder, filename)        
                
                try:
                    y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
                    num_slices = len(y) // samples_per_slice
                    
                    for i in range(num_slices):
                        start = i * samples_per_slice
                        end = start + samples_per_slice
                        y_slice = y[start:end]
                        
                        # Pass the current integer ID into the extraction function
                        slice_features = extract_features_from_slice(y_slice, sr, label=genre, track_id=current_song_id)
                        all_data.append(slice_features)
                    
                    # --- ADDED: After ALL slices for this song are done, increase the ID by 1 for the next song ---
                    current_song_id += 1
                        
                except Exception as e:
                    print(f"  -> Skipped {filename} due to error: {e}")
                    
    df = pd.DataFrame(all_data)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSUCCESS! Dataset saved as {OUTPUT_CSV}")
    print(f"Total Rows: {len(df)}")
    print(f"Total Columns: {len(df.columns)}") 

if __name__ == "__main__":
    build_dataset()

Processing genre: blues...
Processing genre: classical...
Processing genre: country...
Processing genre: disco...
Processing genre: hiphop...
Processing genre: jazz...
  -> Skipped jazz.00054.wav due to error: 
Processing genre: metal...
Processing genre: pop...
Processing genre: reggae...
Processing genre: rock...

SUCCESS! Dataset saved as music_5sec_features(upd).csv
Total Rows: 5985
Total Columns: 65
